## 0. Setup & Sample Dataset

In [1]:
import pandas as pd
import numpy as np

# Create a sample sales dataset
data = {
    "order_id":   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer":   ["Alice", "Bob", "Alice", "Carol", "Bob", "Dave", "Carol", "Alice", "Dave", "Bob"],
    "product":    ["Laptop", "Phone", "Tablet", "Laptop", "Tablet", "Phone", "Laptop", "Phone", "Tablet", "Laptop"],
    "category":   ["Electronics", "Electronics", "Electronics", "Electronics", "Electronics",
                   "Electronics", "Electronics", "Electronics", "Electronics", "Electronics"],
    "quantity":   [1, 2, 1, 1, 3, 2, 1, 1, 2, 1],
    "unit_price": [1200.0, 800.0, 450.0, 1200.0, 450.0, 800.0, 1200.0, 800.0, 450.0, 1200.0],
    "discount":   [0.10, 0.05, 0.0, 0.15, 0.05, 0.0, 0.10, 0.0, 0.05, 0.20],
    "region":     ["North", "South", "East", "North", "West", "South", "East", "North", "West", "South"],
    "date":       ["2024-01-05", "2024-01-12", "2024-02-03", "2024-02-18",
                   "2024-03-07", "2024-03-15", "2024-04-02", "2024-04-20",
                   "2024-05-10", "2024-05-25"],
    "rating":     [4.5, 3.8, None, 4.2, 4.0, None, 3.9, 4.7, 4.1, None]
}

df = pd.DataFrame(data)
print("Dataset created — shape:", df.shape)
df

Dataset created — shape: (10, 10)


,order_id,customer,product,category,quantity,unit_price,discount,region,date,rating
0,101,Alice,Laptop,Electronics,1,1200.0,0.10,North,2024-01-05,4.5
1,102,Bob,Phone,Electronics,2,800.0,0.05,South,2024-01-12,3.8
2,103,Alice,Tablet,Electronics,1,450.0,0.00,East,2024-02-03,NaN
3,104,Carol,Laptop,Electronics,1,1200.0,0.15,North,2024-02-18,4.2
4,105,Bob,Tablet,Electronics,3,450.0,0.05,West,2024-03-07,4.0
5,106,Dave,Phone,Electronics,2,800.0,0.00,South,2024-03-15,NaN
6,107,Carol,Laptop,Electronics,1,1200.0,0.10,East,2024-04-02,3.9
7,108,Alice,Phone,Electronics,1,800.0,0.00,North,2024-04-20,4.7
8,109,Dave,Tablet,Electronics,2,450.0,0.05,West,2024-05-10,4.1
9,110,Bob,Laptop,Electronics,1,1200.0,0.20,South,2024-05-25,NaN


## 1. Exploring the Dataset

In [2]:
print("Shape (rows x cols):", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nFirst 3 rows:")
print(df.head(3))
print("\nLast 3 rows:")
print(df.tail(3))
print("\nBasic statistics:")
print(df.describe())

Shape (rows x cols): (10, 10)

Column dtypes:
order_id        int64
customer       object
product        object
category       object
quantity        int64
unit_price    float64
discount      float64
region         object
date           object
rating        float64
dtype: object

First 3 rows:
   order_id customer product     category  quantity  unit_price  discount  \
0       101    Alice  Laptop  Electronics         1      1200.0      0.10   
1       102      Bob   Phone  Electronics         2       800.0      0.05   
2       103    Alice  Tablet  Electronics         1       450.0      0.00   

  region        date  rating  
0  North  2024-01-05     4.5  
1  South  2024-01-12     3.8  
2   East  2024-02-03     NaN  

Last 3 rows:
   order_id customer product     category  quantity  unit_price  discount  \
7       108    Alice   Phone  Electronics         1       800.0      0.00   
8       109     Dave  Tablet  Electronics         2       450.0      0.05   
9       110      Bob  Lapto

## 2. Handling Missing Values

In [3]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\nTotal missing:", df.isnull().sum().sum())

# Fill missing ratings with column mean
mean_rating = df["rating"].mean()
df["rating"] = df["rating"].fillna(round(mean_rating, 2))
print(f"\nFilled missing ratings with mean: {mean_rating:.2f}")
print("Missing after fill:", df["rating"].isnull().sum())

# Check for duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

Missing values per column:
order_id      0
customer      0
product       0
category      0
quantity      0
unit_price    0
discount      0
region        0
date          0
rating        3
dtype: int64

Total missing: 3

Filled missing ratings with mean: 4.17
Missing after fill: 0

Duplicate rows: 0


## 3. Data Type Conversion

In [4]:
# Convert date column to datetime
df["date"] = pd.to_datetime(df["date"])

# Extract useful date parts
df["month"]      = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%B")
df["quarter"]    = df["date"].dt.quarter

# Convert discount to percentage string for display
df["discount_pct"] = (df["discount"] * 100).astype(int).astype(str) + "%"

print("dtypes after conversion:")
print(df[["date", "month", "quarter", "discount_pct"]].dtypes)
print()
print(df[["date", "month", "month_name", "quarter", "discount_pct"]].head())

dtypes after conversion:
date            datetime64[ns]
month                    int32
quarter                  int32
discount_pct            object
dtype: object

        date  month month_name  quarter discount_pct
0 2024-01-05      1    January        1          10%
1 2024-01-12      1    January        1           5%
2 2024-02-03      2   February        1           0%
3 2024-02-18      2   February        1          15%
4 2024-03-07      3      March        1           5%


## 4. Adding Computed Columns

In [5]:
# Revenue after discount
df["revenue"] = df["quantity"] * df["unit_price"] * (1 - df["discount"])
df["revenue"] = df["revenue"].round(2)

# Discount amount saved
df["discount_saved"] = (df["quantity"] * df["unit_price"] * df["discount"]).round(2)

# Rating category using np.select
conditions = [df["rating"] >= 4.5, df["rating"] >= 4.0, df["rating"] >= 3.5]
choices    = ["Excellent", "Good", "Average"]
df["rating_label"] = np.select(conditions, choices, default="Poor")

print(df[["order_id", "customer", "product", "quantity",
          "unit_price", "discount", "revenue", "discount_saved", "rating_label"]])

   order_id customer product  quantity  unit_price  discount  revenue  \
0       101    Alice  Laptop         1      1200.0      0.10   1080.0   
1       102      Bob   Phone         2       800.0      0.05   1520.0   
2       103    Alice  Tablet         1       450.0      0.00    450.0   
3       104    Carol  Laptop         1      1200.0      0.15   1020.0   
4       105      Bob  Tablet         3       450.0      0.05   1282.5   
5       106     Dave   Phone         2       800.0      0.00   1600.0   
6       107    Carol  Laptop         1      1200.0      0.10   1080.0   
7       108    Alice   Phone         1       800.0      0.00    800.0   
8       109     Dave  Tablet         2       450.0      0.05    855.0   
9       110      Bob  Laptop         1      1200.0      0.20    960.0   

   discount_saved rating_label  
0           120.0    Excellent  
1            80.0      Average  
2             0.0         Good  
3           180.0         Good  
4            67.5         Good 

## 5. Filtering & Selecting Data

In [6]:
# Single condition
laptops = df[df["product"] == "Laptop"]
print("Laptop orders:")
print(laptops[["order_id", "customer", "revenue"]])

# Multiple conditions (AND)
high_value = df[(df["revenue"] > 1000) & (df["discount"] > 0)]
print("\nHigh-value orders with discount:")
print(high_value[["order_id", "customer", "product", "revenue", "discount"]])

# isin() filter
north_south = df[df["region"].isin(["North", "South"])]
print("\nNorth & South orders:", len(north_south))

# Select specific columns with loc
subset = df.loc[df["rating"] >= 4.5, ["customer", "product", "rating", "rating_label"]]
print("\nExcellent rated orders:")
print(subset)

Laptop orders:
   order_id customer  revenue
0       101    Alice   1080.0
3       104    Carol   1020.0
6       107    Carol   1080.0
9       110      Bob    960.0

High-value orders with discount:
   order_id customer product  revenue  discount
0       101    Alice  Laptop   1080.0      0.10
1       102      Bob   Phone   1520.0      0.05
3       104    Carol  Laptop   1020.0      0.15
4       105      Bob  Tablet   1282.5      0.05
6       107    Carol  Laptop   1080.0      0.10

North & South orders: 6

Excellent rated orders:
  customer product  rating rating_label
0    Alice  Laptop     4.5    Excellent
7    Alice   Phone     4.7    Excellent


## 6. Sorting Data

In [7]:
# Sort by single column
print("By revenue (descending):")
print(df.sort_values("revenue", ascending=False)[["order_id","customer","product","revenue"]].head(5))

# Sort by multiple columns
print("\nBy product then revenue:")
print(df.sort_values(["product", "revenue"], ascending=[True, False])[["product","customer","revenue"]].head(6))

# Rank column
df["revenue_rank"] = df["revenue"].rank(ascending=False).astype(int)
print("\nRevenue ranks:")
print(df[["order_id","customer","revenue","revenue_rank"]].sort_values("revenue_rank"))

By revenue (descending):
   order_id customer product  revenue
5       106     Dave   Phone   1600.0
1       102      Bob   Phone   1520.0
4       105      Bob  Tablet   1282.5
0       101    Alice  Laptop   1080.0
6       107    Carol  Laptop   1080.0

By product then revenue:
  product customer  revenue
0  Laptop    Alice   1080.0
6  Laptop    Carol   1080.0
3  Laptop    Carol   1020.0
9  Laptop      Bob    960.0
5   Phone     Dave   1600.0
1   Phone      Bob   1520.0

Revenue ranks:
   order_id customer  revenue  revenue_rank
5       106     Dave   1600.0             1
1       102      Bob   1520.0             2
4       105      Bob   1282.5             3
0       101    Alice   1080.0             4
6       107    Carol   1080.0             4
3       104    Carol   1020.0             6
9       110      Bob    960.0             7
8       109     Dave    855.0             8
7       108    Alice    800.0             9
2       103    Alice    450.0            10


## 7. Grouping & Aggregation

In [8]:
# Revenue per customer
print("Revenue per customer:")
print(df.groupby("customer")["revenue"].sum().sort_values(ascending=False))

# Multiple aggregations per product
print("\nProduct summary:")
product_summary = df.groupby("product").agg(
    total_orders   = ("order_id",  "count"),
    total_revenue  = ("revenue",   "sum"),
    avg_rating     = ("rating",    "mean"),
    total_qty      = ("quantity",  "sum")
).round(2)
print(product_summary)

# Revenue by region
print("\nRevenue by region:")
print(df.groupby("region")["revenue"].agg(["sum", "mean", "count"]).round(2))

# Monthly revenue
print("\nMonthly revenue:")
print(df.groupby(["month", "month_name"])["revenue"].sum().reset_index().sort_values("month"))

Revenue per customer:
customer
Bob      3762.5
Dave     2455.0
Alice    2330.0
Carol    2100.0
Name: revenue, dtype: float64

Product summary:
         total_orders  total_revenue  avg_rating  total_qty
product                                                    
Laptop              4         4140.0        4.19          4
Phone               3         3920.0        4.22          5
Tablet              3         2587.5        4.09          6

Revenue by region:
           sum     mean  count
region                        
East    1530.0   765.00      2
North   2900.0   966.67      3
South   4080.0  1360.00      3
West    2137.5  1068.75      2

Monthly revenue:
   month month_name  revenue
0      1    January   2600.0
1      2   February   1470.0
2      3      March   2882.5
3      4      April   1880.0
4      5        May   1815.0


## 8. Applying Functions

In [9]:
# apply() with lambda
df["revenue_tier"] = df["revenue"].apply(lambda x: "High" if x > 1000 else ("Mid" if x > 500 else "Low"))
print("Revenue tiers:")
print(df[["order_id", "revenue", "revenue_tier"]])

# map() to rename values
region_map = {"North": "N", "South": "S", "East": "E", "West": "W"}
df["region_code"] = df["region"].map(region_map)
print("\nRegion codes:")
print(df[["region", "region_code"]].drop_duplicates())

# applymap / map on full DataFrame (string columns)
str_cols = df[["customer", "product"]].map(str.upper)
print("\nUppercased string columns:")
print(str_cols.head())

Revenue tiers:
   order_id  revenue revenue_tier
0       101   1080.0         High
1       102   1520.0         High
2       103    450.0          Low
3       104   1020.0         High
4       105   1282.5         High
5       106   1600.0         High
6       107   1080.0         High
7       108    800.0          Mid
8       109    855.0          Mid
9       110    960.0          Mid

Region codes:
  region region_code
0  North           N
1  South           S
2   East           E
4   West           W

Uppercased string columns:
  customer product
0    ALICE  LAPTOP
1      BOB   PHONE
2    ALICE  TABLET
3    CAROL  LAPTOP
4      BOB  TABLET


## 9. String Operations

In [10]:
# All pandas str accessor methods
print("Lowercase customers:", df["customer"].str.lower().tolist())
print("Starts with 'A':",    df[df["customer"].str.startswith("A")]["customer"].tolist())
print("Contains 'al':",      df[df["customer"].str.contains("al", case=False)]["customer"].tolist())
print("Name lengths:",        df["customer"].str.len().tolist())

# String concatenation
df["order_label"] = "Order-" + df["order_id"].astype(str) + " | " + df["customer"]
print("\nOrder labels:")
print(df["order_label"].tolist())

Lowercase customers: ['alice', 'bob', 'alice', 'carol', 'bob', 'dave', 'carol', 'alice', 'dave', 'bob']
Starts with 'A': ['Alice', 'Alice', 'Alice']
Contains 'al': ['Alice', 'Alice', 'Alice']
Name lengths: [5, 3, 5, 5, 3, 4, 5, 5, 4, 3]

Order labels:
['Order-101 | Alice', 'Order-102 | Bob', 'Order-103 | Alice', 'Order-104 | Carol', 'Order-105 | Bob', 'Order-106 | Dave', 'Order-107 | Carol', 'Order-108 | Alice', 'Order-109 | Dave', 'Order-110 | Bob']


## 10. NumPy Array Operations

In [11]:
revenue_arr = df["revenue"].to_numpy()
print("Revenue array:", revenue_arr)

# Basic stats
print(f"\nMean:    {np.mean(revenue_arr):.2f}")
print(f"Median:  {np.median(revenue_arr):.2f}")
print(f"Std Dev: {np.std(revenue_arr):.2f}")
print(f"Min/Max: {np.min(revenue_arr):.2f} / {np.max(revenue_arr):.2f}")

# Percentiles
p25, p50, p75 = np.percentile(revenue_arr, [25, 50, 75])
print(f"25th percentile: {p25:.2f}")
print(f"50th percentile: {p50:.2f}")
print(f"75th percentile: {p75:.2f}")

# Boolean masking
above_mean = revenue_arr[revenue_arr > np.mean(revenue_arr)]
print("\nRevenues above mean:", above_mean)

# Normalize (min-max scaling)
normalized = (revenue_arr - revenue_arr.min()) / (revenue_arr.max() - revenue_arr.min())
print("Normalized (0-1):", normalized.round(3))

Revenue array: [1080.  1520.   450.  1020.  1282.5 1600.  1080.   800.   855.   960. ]

Mean:    1064.75
Median:  1050.00
Std Dev: 323.51
Min/Max: 450.00 / 1600.00
25th percentile: 881.25
50th percentile: 1050.00
75th percentile: 1231.88

Revenues above mean: [1080.  1520.  1282.5 1600.  1080. ]
Normalized (0-1): [0.548 0.93  0.    0.496 0.724 1.    0.548 0.304 0.352 0.443]


## 11. Final Summary Report

In [12]:
print("=" * 45)
print("         SALES DATA SUMMARY REPORT")
print("=" * 45)
print(f"Total Orders          : {len(df)}")
print(f"Total Revenue         : ${df['revenue'].sum():,.2f}")
print(f"Average Order Value   : ${df['revenue'].mean():,.2f}")
print(f"Highest Order Revenue : ${df['revenue'].max():,.2f}")
print(f"Total Discount Saved  : ${df['discount_saved'].sum():,.2f}")
print(f"Average Rating        : {df['rating'].mean():.2f}")
print()
print("--- Revenue by Customer ---")
for cust, rev in df.groupby("customer")["revenue"].sum().sort_values(ascending=False).items():
    print(f"  {cust:<10}: ${rev:,.2f}")
print()
print("--- Top Product by Revenue ---")
top_product = df.groupby("product")["revenue"].sum().idxmax()
top_rev     = df.groupby("product")["revenue"].sum().max()
print(f"  {top_product} — ${top_rev:,.2f}")
print()
print("--- Revenue Tiers ---")
print(df["revenue_tier"].value_counts().to_string())
print("=" * 45)

         SALES DATA SUMMARY REPORT
Total Orders          : 10
Total Revenue         : $10,647.50
Average Order Value   : $1,064.75
Highest Order Revenue : $1,600.00
Total Discount Saved  : $852.50
Average Rating        : 4.17

--- Revenue by Customer ---
  Bob       : $3,762.50
  Dave      : $2,455.00
  Alice     : $2,330.00
  Carol     : $2,100.00

--- Top Product by Revenue ---
  Laptop — $4,140.00

--- Revenue Tiers ---
revenue_tier
High    6
Mid     3
Low     1
